# REROUTE 가상 데이터 분석

> **중요:** 실제 고객 조사나 운영 기록이 아닙니다. 제품 판단 기준을 미리 점검하려고 같은 결과를 재현할 수 있는 난수 초기값으로 만든 가상 데이터입니다.


## 핵심 요약

- 판단: 가상 데이터만으로 후속 개발 여부를 결정하지 않습니다.
- 근거: 행동 기준은 방향성을 보이지만 유료 파일럿 참여 의향 기준의 불확실성이 큽니다.
- 검증안: 실제 사업 환경에서는 사업팀이 가격 수용도, 구매 결정권자와 계약 주체를 확인합니다. 개인 프로젝트에서는 참여 기업을 모집하거나 영업하지 않았습니다.


## 전제와 방법

가상 사용자 10명과 가상 기업 5곳을 가정했습니다. 각 전환 확률을 베타 분포에서 다시 뽑아 표본 오차와 가정 오차를 함께 반영했습니다. 기준 시나리오는 50,000회, 보수 및 낙관 시나리오는 각각 20,000회 반복했습니다.


In [1]:
from pathlib import Path
import csv
import json

root = Path.cwd()
if not (root / 'analysis').exists():
    root = root.parent
results_path = root / 'analysis/generated/validation-simulation-results.json'
cohort_path = root / 'analysis/generated/validation-simulation-cohort.csv'
results = json.loads(results_path.read_text(encoding='utf-8'))
with cohort_path.open(encoding='utf-8') as handle:
    cohort = list(csv.DictReader(handle))

assert results['synthetic'] is True
print(json.dumps({
    'artifact_type': results['artifact_type'],
    'synthetic': results['synthetic'],
    'seed': results['seed'],
    'base_iterations': results['base']['iterations'],
    'cohort_rows': len(cohort),
}, ensure_ascii=False, indent=2))


{
  "artifact_type": "synthetic_validation_simulation",
  "synthetic": true,
  "seed": 20260901,
  "base_iterations": 50000,
  "cohort_rows": 15
}


## Data Validation

저장된 예시 결과 CSV를 별도로 다시 집계해 JSON 결과와 일치하는지 확인합니다.


In [2]:
users = [row for row in cohort if row['record_type'] == 'synthetic_user']
companies = [row for row in cohort if row['record_type'] == 'synthetic_company']
recomputed = {
    'dashboard_viewed': sum(int(row['dashboard_viewed']) for row in users),
    'bids_opened': sum(int(row['bids_opened']) for row in users),
    'recalculation_opened': sum(int(row['recalculation_opened']) for row in users),
    'confirmation_opened': sum(int(row['confirmation_opened']) for row in users),
    'pilot_interest': sum(int(row['pilot_interest']) for row in companies),
    'paid_pilot': sum(int(row['paid_pilot']) for row in companies),
}
stored = results['base']['representative']['counts']
assert len(users) == 10 and len(companies) == 5
assert recomputed == stored
print('PASS: CSV 재집계와 JSON 대표 지표가 일치합니다.')
print(json.dumps(recomputed, ensure_ascii=False, indent=2))


PASS: CSV 재집계와 JSON 대표 지표가 일치합니다.
{
  "dashboard_viewed": 10,
  "bids_opened": 7,
  "recalculation_opened": 4,
  "confirmation_opened": 3,
  "pilot_interest": 3,
  "paid_pilot": 1
}


## Results

기준 시나리오의 성공 기준별 충족 확률을 확인합니다.


In [3]:
labels = {
    'bids_opened': '인수처 확인 자료 검토',
    'recalculation_opened': '조건 탐색',
    'confirmation_opened': '배분안 확정 화면 진입',
    'pilot_interest': '파일럿 참여 의향',
    'paid_pilot': '유료 파일럿 참여 의향',
    'all_criteria': '모든 기준 동시 충족',
}
probabilities = results['base']['pass_probabilities']
print('| 기준 | 충족 추정 확률 |')
print('| --- | ---: |')
for key in labels:
    print(f"| {labels[key]} | {probabilities[key]:.1f}% |")
print(f"\n후속 개발 검토 기준 동시 충족: {probabilities['investment_rule']:.1f}%")


| 기준 | 충족 추정 확률 |
| --- | ---: |
| 인수처 확인 자료 검토 | 77.7% |
| 조건 탐색 | 75.2% |
| 배분안 확정 화면 진입 | 78.4% |
| 파일럿 참여 의향 | 63.5% |
| 유료 파일럿 참여 의향 | 39.3% |
| 모든 기준 동시 충족 | 18.8% |

후속 개발 검토 기준 동시 충족: 30.9%


## 결론

1. 가상 데이터만으로 후속 개발 여부를 판단하지 않습니다.
2. 개인 프로젝트에서는 참여 기업을 모집하거나 영업하지 않았고, 계약도 체결하지 않았습니다.
3. 실제 사업 환경에서 사업팀이 고객 검증을 진행한다면 기업 5곳 중 2곳 이상에서 유료 파일럿 참여 의향을 확인하고, 테스트 사용자 중 20% 이상이 배분안 확정 화면에 도달했을 때 후속 개발을 검토합니다.

### 한계

이 결과는 실제 사용자, 고객사, 계약, 매출을 대체하지 않으며 시장 수요나 인과 효과를 증명하지 않습니다.
